# Ouysse - Calès 
Deux sondes : **CTD** (Diver autonome : niveau, conductivité, température), **OTT** (sonde CTD de la centrale : niveau, conductivité, température).

## 1. Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import ouysse
from ouysse import *        

## 2. Chemins d'accès

In [ ]:
BASE        = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Ouysse - Calès\Gaetan"
CTD_PATH    = os.path.join(BASE, r"Données brutes\CTD")
OTT_PATH    = os.path.join(BASE, r"Données brutes\OTT")
BARO_PATH   = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\1 - Données BARO\Gourdon baro\Patm Calès et Thémines.xlsx"
PLUIE_PATH  = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Pluie_BV_Ouysse.csv"

OLDDATA_PATH     = os.path.join(BASE, "OuysseCales_consolide_old.xlsx")
UTC_CTD_PATH     = os.path.join(BASE, r"UTC_CTD.xlsx")
PUNCTUAL_NIVEAU  = os.path.join(BASE, "punctual_measurements_niveau.xlsx")
PUNCTUAL_CONDUCT = os.path.join(BASE, "punctual_measurements_conducti.xlsx")
SORTIE_CONSOLIDE = os.path.join(BASE, "OuysseCales_consolide.xlsx")
SORTIE_FINALE    = os.path.join(BASE, "OuysseCales_final.xlsx")
SORTIE_SVG       = os.path.join(BASE, "Graphes.svg")

PREFIXE_CTD = "Ouysse"
BARO_COL    = "Patm Ouysse Calès [hPa]"
PAS         = "1h"

## 3. CTD : lecture, UTC et compensation barométrique

`lire_CTD` encaisse les pièges du format Diver (en-tête à une ligne variable, pied
`END OF DATA`, virgules décimales, mS/cm ou µS/cm). La table UTC nomme les fichiers
exactement, sinon la campagne est ignorée et le message la nomme.

In [ ]:
metadata = pd.read_excel(UTC_CTD_PATH)
baro = pd.read_excel(BARO_PATH)[["DATE", BARO_COL]]
baro["DATE"] = pd.to_datetime(baro["DATE"], errors="coerce")
baro = baro.dropna(subset=["DATE"]).drop_duplicates("DATE")

files = []
for nom in fichiers(CTD_PATH, PREFIXE_CTD):
    try:
        CTD, decalage = en_utc(lire_CTD(nom, CTD_PATH, PAS), nom, metadata)
    except Exception as e:
        print(f"  IGNORÉ  {nom} : {e}")
        continue
    m = pd.merge(CTD, baro, left_on="Date/time", right_on="DATE", how="left")
    m["Niveau_(cm)"] = m["Pression[cmH2O]"] - m[BARO_COL] * HPA_EN_CMH2O
    m = m.rename(columns={"Température[°C]": "Temp_(°C)"})
    files.append(m[["Date/time", "Niveau_(cm)", "Cond_(µS/cm)", "Temp_(°C)"]])
    print(f"  {nom:45s} UTC+{decalage:g} vers UTC   ({len(CTD)} lignes)")

merge_ctd_df = pd.concat(files, ignore_index=True).sort_values("Date/time", kind="stable")
merge_ctd_df["DATE"] = merge_ctd_df["Date/time"]
print(f"\n{len(files)} campagne(s), {len(merge_ctd_df)} enregistrements en UTC.")

## 4. Raccordement à l'ancienne chronique

L'ancien fichier consolidé et les campagnes récentes sont la **même sonde CTD**, séparées
par un trou d'exploitation : le décalage est mesuré à la jonction et appliqué aux
campagnes, pour que la chronique soit continue.

In [ ]:
#: L'ancien consolidé n'emploie pas toujours les noms de colonnes du notebook.
RENOMMAGE_OLD = {
    "Niveau": "Niveau_CTD_(cm)",
    "Conducti": "Cond_CTD_(µS/cm)",
    "Temp": "Temp _CTD(°C)",
}

olddata_df = pd.read_excel(OLDDATA_PATH)
olddata_df["DATE"] = pd.to_datetime(olddata_df["DATE"], errors="coerce")
olddata_df = (olddata_df.rename(columns=RENOMMAGE_OLD).dropna(subset=["DATE"])
              .sort_values("DATE"))

merge_ctd_df = raccorder_campagnes(olddata_df, merge_ctd_df, [
    ("Niveau", "Niveau_CTD_(cm)", "Niveau_(cm)", "cm"),
    ("Conductivité", "Cond_CTD_(µS/cm)", "Cond_(µS/cm)", "µS/cm")])

## 5. Centrale OTT

In [ ]:
VOIES_OTT = NOMS_OTT_CTD

merge_ott_df = (pd.concat([lire_OTT(nom, OTT_PATH, VOIES_OTT, PAS)
                           for nom in fichiers(OTT_PATH, "")], ignore_index=True)
                .groupby("DATE", as_index=False).median(numeric_only=True).sort_values("DATE"))
print(f"{len(merge_ott_df)} pas de temps, du {merge_ott_df['DATE'].min():%d/%m/%Y} "
      f"au {merge_ott_df['DATE'].max():%d/%m/%Y}")

## 6. Assemblage : une colonne par sonde

In [ ]:
COLONNES_CTD   = ["Niveau_CTD_(cm)", "Cond_CTD_(µS/cm)", "Temp _CTD(°C)"]

full_data = sur_grille([
    empiler([olddata_df,
             merge_ctd_df.rename(columns={"Niveau_(cm)": "Niveau_CTD_(cm)",
                                          "Cond_(µS/cm)": "Cond_CTD_(µS/cm)",
                                          "Temp_(°C)": "Temp _CTD(°C)"})], COLONNES_CTD),
    empiler([merge_ott_df], list(VOIES_OTT.values())),
], PAS)

print("Hors gamme physique :")
full_data = appliquer_gammes(full_data)

#: Les sondes de chaque grandeur : {grandeur: {sonde: colonne}}.
SONDES = {
    "Niveau_(cm)":        {"OTT": "Niveau_CTDOTT_(cm)", "CTD": "Niveau_CTD_(cm)"},
    "Conductivité":       {"OTT": "Cond_CTDOTT_(µS/cm)", "CTD": "Cond_CTD_(µS/cm)"},
    "Température":        {"OTT": "Temp_CTDOTT_(°C)", "CTD": "Temp _CTD(°C)"},
}
PARAMETRES = list(SONDES)

#: Ordre de préférence automatique : à chaque pas, la première sonde qui mesure.
ORDRE = ["OTT", "CTD"]

BRUT = full_data.copy()
full_data.to_excel(SORTIE_CONSOLIDE)
print(f"\nfichier fusionné : {SORTIE_CONSOLIDE}")

## 7. Corrections capteur

Les deux seules corrections qui portent sur une **sonde**, avant toute fusion.

`VOIES_ECARTEES` met des mesures à l'écart : la voie est retirée avant la fusion, donc
une autre sonde prend le relais si elle mesure ; s'il n'y en a pas, la lacune reste et
l'interpolation ne comblera pas plus de 12 h.

`CALAGES_SONDE` déplace une sonde entière, ou son passé, ou son avenir.

In [ ]:
#: (début, fin, colonne, motif) : mesures mises à l'écart, toutes grandeurs.
VOIES_ECARTEES = [
    ("2024-10-02 10:00", "2024-12-10 16:00", "Cond_CTD_(µS/cm)", "valeurs aberrantes"),
    ("2026-02-10 20:00", "2026-02-15 7:00", "Cond_CTD_(µS/cm)", "valeurs aberrantes"),

]

#: (date d'ancrage, sonde, grandeur, décalage, sens) : ajustements manuels.
#: sens = "amont" (avant la date) | "aval" (à partir de la date) | "tout".
CALAGES_SONDE = [

]
print("Voies écartées :")
CORRIGE = ecarter(BRUT.copy(), VOIES_ECARTEES)
def voies_calees(grandeur):
    """Les séries des sondes d'une grandeur, écarts et calages appliqués."""
    series = {}
    for sonde, col in SONDES[grandeur].items():
        s = CORRIGE[col]
        for date, cible, valeur, sens in CALAGES_SONDE:
            if cible not in CORRIGE.columns:
                raise ValueError(f"CALAGES_SONDE : colonne inconnue {cible!r}")
            if cible == col:
                s = decaler(s, date, valeur, sens)
                print(f"  {col} : {valeur:+.2f} en {sens} du "
                      f"{pd.to_datetime(date):%d/%m/%Y %H:%M}")
        series[sonde] = s
    return series

## 8. Comparaison des sources

À lancer pour juger quelle sonde garder sur une période, avant d'écrire une période
imposée dans les cellules suivantes.

In [ ]:
PARAMETRE = "Conductivité"   # "Niveau_(cm)", "Conductivité", "Température"

graphe_sondes({sonde: BRUT[col] for sonde, col in SONDES[PARAMETRE].items()},
              titre=f"{PARAMETRE} : les sondes disponibles", ylab=PARAMETRE)

## 9. Niveau

La sonde est choisie automatiquement, dans l'ordre `ORDRE` déclaré à l'assemblage.
`SONDE_PRIORITAIRE_NIVEAU` sert à imposer une autre sonde sur une période précise. Les
périodes retenues sont affichées, avec le recalage appliqué à chaque changement.

In [ ]:
#: (début, fin, sonde imposée) : sort du choix automatique sur cette période.
SONDE_PRIORITAIRE_NIVEAU = [
    ("2024-12-10 16:00", "2026-02-11 09:00", "CTD"),
]
points_niveau = lire_points(PUNCTUAL_NIVEAU)

print("Calages de sonde :")
voies = voies_calees("Niveau_(cm)")
print("Périodes retenues :")
avant, source = fusionner(voies, choisir_sondes(voies, ORDRE, SONDE_PRIORITAIRE_NIVEAU), "cm")

print("Points de contrôle :")
niveau = caler(avant, points_niveau, "Hauteur (cm)")
full_data["Niveau_(cm)"], full_data["Niveau_(cm)_source"] = niveau, source
print("Pas de temps par sonde :", source.value_counts().to_dict())

graphe_sondes(voies, avant, niveau, titre="Niveau", ylab="Niveau (cm)",
              points=points_niveau, col_point="Hauteur (cm)")

## 10. Conductivité

In [ ]:
#: (début, fin, sonde imposée) : sort du choix automatique sur cette période.
SONDE_PRIORITAIRE_COND = [

]
points_cond = lire_points(PUNCTUAL_CONDUCT)

print("Calages de sonde :")
voies = voies_calees("Conductivité")
print("Périodes retenues :")
avant, source = fusionner(voies, choisir_sondes(voies, ORDRE, SONDE_PRIORITAIRE_COND), "µS/cm")

print("Points de contrôle :")
cond = caler(avant, points_cond, "Conductivité")
print("Pas de temps par sonde :", source.value_counts().to_dict())

graphe_sondes(voies, avant, cond, titre="Conductivité", ylab="Conductivité (µS/cm)",
              points=points_cond, col_point="Conductivité")

### Filtre IQR et lissage

Post-traitement appliqué **après** la fusion et le calage sur les points de contrôle :
c'est cette chronique nettoyée qui alimente `full_data` et le fichier final.

In [ ]:
FENETRE_IQR, K_IQR = "500h", 0.8   # k = 0 : pas de filtre
LISSAGE_H = 6                     # 0 = pas de lissage ; sinon médiane glissante, en heures

cond_iqr = filtre_iqr(cond, FENETRE_IQR, K_IQR, lissage_h=LISSAGE_H)
full_data["Conductivité"], full_data["Conductivité_source"] = cond_iqr, source
full_data["Conductivité_Moyenne_Mobile"] = cond_iqr.rolling("6h", center=True).mean()

graphe([(cond, "avant IQR et lissage", "darkorange"),
        (cond_iqr, "après IQR et lissage", "black")],
       titre="Conductivité", ylab="Conductivité (µS/cm)")

## 11. Température et autres paramètres

Choix automatique, pas de calage sur points de contrôle. Les voies défaillantes ont
déjà été écartées à la cellule des corrections capteur.

In [ ]:
#: {grandeur: [(début, fin, sonde imposée)]} pour sortir du choix automatique.
EXCEPTIONS = {

}

for grandeur in ["Température"]:
    print(f"{grandeur} :")
    voies = voies_calees(grandeur)
    full_data[grandeur], full_data[f"{grandeur}_source"] = fusionner(
        voies, choisir_sondes(voies, ORDRE, EXCEPTIONS.get(grandeur, [])))
    print(f"  {full_data[grandeur].notna().sum()} pas  "
          f"{full_data[f'{grandeur}_source'].value_counts().to_dict()}")

## 12. Débit

Deux branches raccordees a H = 1 m, hauteur en metres dans les deux formules.
Le débit est calculé après les corrections du niveau, puis recalculé sur le
niveau interpolé.

In [ ]:
SEUIL_H = 100.0     # cm, raccord des deux branches de la courbe de tarage

def debit(H):
    """Débit en m3/s à partir du niveau en cm, jamais négatif."""
    h = pd.to_numeric(H, errors="coerce").astype("float64")
    Q = np.where(h >= SEUIL_H, 10.227 * (h / 100)**2 + 7.3348 * (h / 100) - 10,
                 7.7563 * (h / 100)**7.6404)
    return pd.Series(np.where(h.isna(), np.nan, np.maximum(Q, 0.0)), index=H.index)

full_data["Q_(m3/s)"] = debit(full_data["Niveau_(cm)"])
display(full_data[["Niveau_(cm)", "Q_(m3/s)"]].describe().round(3))

## 13. Cote NGF, interpolation et statuts

Les lacunes de moins de 12 h sont comblées. `Statut_<grandeur>` dit si la valeur est
mesurée, interpolée ou manquante.

In [ ]:
NIVEAU_NGF = None       # cote du zéro de l'échelle, None si elle n'est pas connue
MAX_TROU_H = 12

full_data = interpoler_avec_statut(full_data, PARAMETRES, MAX_TROU_H, PAS)

if NIVEAU_NGF is not None:
    full_data["Niveau_(mNGF)"] = NIVEAU_NGF + full_data["Niveau_(cm)"] / 100
    full_data["Statut_Niveau_(mNGF)"] = full_data["Statut_Niveau_(cm)"]
    print(f"Zéro de l'échelle à {NIVEAU_NGF:.4f} m NGF")
full_data["Q_(m3/s)"] = debit(full_data["Niveau_(cm)"])   # sur le niveau interpolé
full_data["Statut_Q_(m3/s)"] = full_data["Statut_Niveau_(cm)"]

display(pd.DataFrame({c: full_data[f"Statut_{c}"].value_counts()
                      for c in PARAMETRES}).fillna(0).astype(int).T)

## 14. Sauvegarde et graphe de synthèse

Un paramètre par grandeur, avec son statut. Le détail capteur par capteur, et la sonde
retenue à chaque pas, restent dans le fichier consolidé écrit à l'assemblage. La version
de `ouysse-hydro` est affichée : c'est elle qui dit avec quel code la chronique a été
produite.

In [ ]:
finaux = [c for c in PARAMETRES + ["Q_(m3/s)", "Niveau_(mNGF)"] if c in full_data]
colonnes = [c for p in finaux for c in (p, f"Statut_{p}") if c in full_data]
sortie = full_data[colonnes].copy()
sortie.attrs["ouysse"] = ouysse.__version__
sortie.to_excel(SORTIE_FINALE)
print(f"{SORTIE_FINALE} : {len(sortie)} pas x {len(colonnes)} colonnes "
      f"(ouysse-hydro {ouysse.__version__})")

graphe_synthese(full_data, "Q_(m3/s)", "Débit (m3/s)", pluie=PLUIE_PATH, sortie=SORTIE_SVG,marge_jours=15)

## 15. Contrôle des interpolations

In [ ]:
graphe_statuts(full_data, finaux)